In [ ]:
import requests
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("USE SCHEMA bronze")

CEMAC_COUNTRIES = ["CMR", "CAF", "TCD", "COG", "GNQ", "GAB"]
INDICATOR_CODE = "NY.GDP.MKTP.CD"
DATE_RANGE = "1990:2024"

print(f"Catalog: cemac_ecowas_aes_trade")
print(f"Schema: bronze")
print(f"Target countries: {CEMAC_COUNTRIES}")
print(f"Indicator: {INDICATOR_CODE} (GDP, current US$)")
print(f"Date range: {DATE_RANGE}")

In [ ]:
country_list = ";".join(CEMAC_COUNTRIES)
url = f"https://api.worldbank.org/v2/country/{country_list}/indicator/{INDICATOR_CODE}"
params = {"format": "json", "date": DATE_RANGE, "per_page": 1000}

print(f"Requesting: {url}")
print(f"Params: {params}")
print()

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

data = response.json()
metadata = data[0]
observations = data[1] if len(data) > 1 else []

print(f"HTTP status: {response.status_code}")
print(f"Pages: {metadata.get('pages')}, total observations: {metadata.get('total')}")
print(f"Observations actually returned: {len(observations)}")
print()
print("First observation (raw):")
print(observations[0] if observations else "NONE")

In [ ]:
# Record the current UTC time for data loading timestamp
loaded_at = datetime.now(timezone.utc)
rows = []

# Transform each observation into a Row object for Spark DataFrame
for obs in observations:
    rows.append(Row(
        country_iso3=obs["countryiso3code"],  # ISO3 country code
        country_name=obs["country"]["value"],  # Country name
        indicator_code=obs["indicator"]["id"],  # Indicator code
        indicator_name=obs["indicator"]["value"],  # Indicator name
        year=int(obs["date"]),  # Year of observation
        value=obs["value"],  # GDP value
        unit=obs.get("unit", ""),  # Unit (if available)
        obs_status=obs.get("obs_status", ""),  # Observation status (if available)
        source="worldbank_data360",  # Data source
        loaded_at=loaded_at,  # Timestamp when loaded
    ))

# Create Spark DataFrame from the list of Row objects
df = spark.createDataFrame(rows)

# Print DataFrame row count
print(f"DataFrame created with {df.count()} rows")
print()
# Print DataFrame schema
df.printSchema()
print()
# Show first 5 rows of the DataFrame
df.show(210)

In [ ]:
df.write.mode("overwrite").saveAsTable("bronze.data360_raw")

print("Write complete.")
print()

result = spark.sql("""
    SELECT COUNT(*) AS row_count,
           COUNT(value) AS non_null_values,
           MIN(year) AS earliest_year,
           MAX(year) AS latest_year,
           COUNT(DISTINCT country_iso3) AS countries_present
    FROM bronze.data360_raw
""")

result.show()

In [ ]:
# Analyze and display GDP data for CEMAC countries, including 2020 GDP, Cameroon's GDP trend, and Delta table history

print("CEMAC GDP in 2020 (latest pre-pandemic year with broad coverage):")
print()

spark.sql("""
    SELECT country_name,
           year,
           ROUND(value / 1e9, 2) AS gdp_usd_billions
    FROM bronze.data360_raw
    WHERE year = 2020
      AND value IS NOT NULL
    ORDER BY value DESC
""").show()

print()
print("Cameroon GDP trajectory (every 5 years):")
print()

spark.sql("""
    SELECT year,
           ROUND(value / 1e9, 2) AS gdp_usd_billions
    FROM bronze.data360_raw
    WHERE country_iso3 = 'CMR'
      AND year IN (1990, 1995, 2000, 2005, 2010, 2015, 2020, 2023, 2024)
    ORDER BY year
""").show()

print()
print("Delta table history (proves time travel works):")
print()

spark.sql("DESCRIBE HISTORY bronze.data360_raw").select("version", "timestamp", "operation").show(truncate=False)

In [ ]:
print("2024 coverage check:")
spark.sql("""
    SELECT country_name,
           year,
           CASE WHEN value IS NULL THEN 'missing'
                ELSE CONCAT('$', ROUND(value/1e9, 2), 'B')
           END AS gdp_2024
    FROM bronze.data360_raw
    WHERE year = 2024
    ORDER BY country_name
""").show(truncate=False)